# Landscape Image Classification — Intel Image Classification

**Tahap 1 dan Tahap 2: perencanaan eksperimen, audit dataset, preprocessing, training, dan evaluasi**

Proyek membandingkan empat eksperimen:

1. **E1 — Custom CNN tanpa augmentasi** (baseline kontrol).
2. **E2 — Custom CNN dengan augmentasi**.
3. **E3 — MobileNetV2 transfer learning / feature extraction**.
4. **E4 — MobileNetV2 fine-tuning**.

Dataset: Intel Image Classification, dengan enam kelas `buildings`, `forest`, `glacier`, `mountain`, `sea`, dan `street`.

- Halaman sumber Kaggle: https://www.kaggle.com/datasets/puneet6060/intel-image-classification
- Mirror publik untuk unduh otomatis: https://huggingface.co/datasets/sfarrukhm/intel-image-classification

> **Tidak perlu upload `kaggle.json`, tidak perlu klik Choose Files, dan tidak perlu upload dataset secara manual.** Notebook mengunduh salinan publik dataset secara otomatis dari Hugging Face. Dataset tetap diatribusikan ke Intel / distribusi Kaggle asal.

Notebook ini dijalankan di **Google Colab dengan GPU**. Training belum dianggap berhasil sebelum seluruh cell selesai, file model dan metrik benar-benar tersimpan. Jangan menulis angka akurasi ke laporan sebelum diperoleh dari eksekusi nyata.


## 0. Protokol eksperimen dan keputusan metodologis

- Dataset berlabel yang digunakan adalah `seg_train` dan `seg_test`. `seg_pred` tidak memiliki label, sehingga tidak digunakan untuk training, validation, atau penghitungan metrik.
- `seg_train` dibagi secara **stratified 80:20** menjadi training dan validation dengan `random_state=42`.
- `seg_test` dipertahankan sebagai holdout test dan hanya dievaluasi setelah konfigurasi/eksperimen ditetapkan.
- Semua eksperimen memakai split yang sama, input RGB 150×150, batch size 32, dan enam label kelas yang sama.
- Augmentasi hanya aktif pada training; validation dan test tidak diaugmentasi.
- Custom CNN melakukan rescaling ke 0–1 di dalam model. MobileNetV2 melakukan rescaling ke −1…1 di dalam model.
- Distribusi kelas aktual diperiksa setelah download. Tidak menggunakan oversampling atau class weighting pada eksperimen utama kecuali audit menunjukkan alasan metodologis yang kuat.
- Model final dipilih menggunakan **validation Macro F1**, bukan test. Test set hanya digunakan untuk evaluasi akhir.
- Seed eksperimen: 42.

### Catatan fairness
Perbandingan utama Custom CNN dengan MobileNetV2 menggunakan preprocessing yang konsisten, data split yang sama, dan test set yang sama. E1 vs E2 mengukur pengaruh augmentasi pada Custom CNN; E3 vs E4 mengukur pengaruh fine-tuning pada MobileNetV2.


In [ ]:
# 1. Instalasi library pendukung untuk Google Colab
%pip -q install --upgrade huggingface_hub hf_xet scikit-learn seaborn

import os, sys, json, math, random, hashlib, shutil, zipfile, gc, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)
from huggingface_hub import snapshot_download

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
if not tf.config.list_physical_devices("GPU"):
    print("PERINGATAN: GPU tidak terdeteksi. Colab: Runtime > Change runtime type > T4 GPU (jika tersedia).")


## 2. Unduh dataset secara otomatis (tanpa Kaggle token)

Untuk menghilangkan langkah upload `kaggle.json`, notebook menggunakan **mirror publik** Intel Image Classification di Hugging Face. Mirror tersebut memuat folder `data/seg_train` dan `data/seg_test` dengan enam folder kelas. Notebook hanya mengunduh data berlabel yang dibutuhkan untuk eksperimen.

**Yang dilakukan cell berikut:**
1. Mengunduh dataset langsung ke runtime Colab.
2. Melewati download jika folder berlabel sudah ada.
3. Tidak meminta file dari komputer pengguna.

Atribusi dataset tetap merujuk ke Intel Image Classification dan halaman distribusi Kaggle asal.


In [ ]:
from google.colab import drive

DATA_ROOT = Path("/content/intel_landscape_data")
HF_LOCAL_DIR = DATA_ROOT / "huggingface_mirror"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
HF_DATASET_REPO = "sfarrukhm/intel-image-classification"

# Cek apakah folder train dan test yang benar sudah tersedia.
existing_train = list(DATA_ROOT.rglob("seg_train"))
existing_test = list(DATA_ROOT.rglob("seg_test"))
if not (existing_train and existing_test):
    print("Mengunduh Intel Image Classification otomatis dari mirror publik Hugging Face...")
    snapshot_download(
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
        local_dir=str(HF_LOCAL_DIR),
        allow_patterns=[
            "data/seg_train/**",
            "data/seg_test/**",
        ],
    )
else:
    print("Folder seg_train dan seg_test sudah tersedia; download dilewati.")

print("Download selesai.")
print("Folder sumber:", HF_LOCAL_DIR)
print("Struktur tingkat atas:")
for p in sorted(DATA_ROOT.iterdir()):
    print("-", p.name, "(folder)" if p.is_dir() else f"({p.stat().st_size:,} bytes)")


## 3. Menemukan folder dataset dan memeriksa struktur

Kaggle dapat membungkus folder dengan satu tingkat direktori tambahan. Fungsi berikut mencari direktori split yang benar berdasarkan keberadaan keenam subfolder kelas.

In [ ]:
CLASS_NAMES = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def locate_split_dir(root: Path, split_name: str) -> Path:
    candidates = [p for p in root.rglob(split_name) if p.is_dir()]
    for candidate in candidates:
        if all((candidate / class_name).is_dir() for class_name in CLASS_NAMES):
            return candidate
    raise FileNotFoundError(
        f"Folder {split_name} dengan enam subfolder kelas tidak ditemukan di {root}. "
        "Periksa kembali arsip dataset yang diunduh."
    )

train_dir = locate_split_dir(DATA_ROOT, "seg_train")
test_dir = locate_split_dir(DATA_ROOT, "seg_test")
print("Train directory:", train_dir)
print("Test directory :", test_dir)

# Folder prediksi biasanya tidak memiliki subfolder label. Opsional, tidak dipakai pada supervised learning.
pred_candidates = [p for p in DATA_ROOT.rglob("seg_pred") if p.is_dir()]
pred_dir = pred_candidates[0] if pred_candidates else None
print("Prediction directory:", pred_dir)

# Validasi struktur sumber: setiap split supervised wajib memiliki enam folder kelas.
assert all((train_dir / name).is_dir() for name in CLASS_NAMES), "Struktur seg_train tidak lengkap."
assert all((test_dir / name).is_dir() for name in CLASS_NAMES), "Struktur seg_test tidak lengkap."


## 4. Audit fisik gambar: jumlah file, file rusak, dan duplikasi

Audit dilakukan pada seluruh gambar berlabel (`seg_train` dan `seg_test`). File gambar yang tidak dapat dibaca dikeluarkan dari manifest eksperimen dan dicatat. Duplikasi identik lintas train-test diperiksa; salinan di sisi training dikeluarkan agar tidak terjadi kebocoran identik ke holdout test. Duplikasi lain dilaporkan agar dapat dipertimbangkan saat interpretasi hasil.

In [ ]:
def collect_image_records(split_dir: Path, split_name: str) -> pd.DataFrame:
    rows = []
    for label, class_name in enumerate(CLASS_NAMES):
        class_dir = split_dir / class_name
        for path in sorted(class_dir.rglob("*")):
            if path.is_file() and path.suffix.lower() in IMAGE_EXTS:
                rows.append({
                    "filepath": str(path),
                    "split": split_name,
                    "class_name": class_name,
                    "label": label,
                })
    return pd.DataFrame(rows)

raw_train_df = collect_image_records(train_dir, "train")
raw_test_df = collect_image_records(test_dir, "test")

print("Jumlah file awal train:", len(raw_train_df))
print("Jumlah file awal test :", len(raw_test_df))
print("Jumlah file awal total:", len(raw_train_df) + len(raw_test_df))

bad_files = []
valid_hashes = []
all_labeled = pd.concat([raw_train_df, raw_test_df], ignore_index=True)

for row in all_labeled.itertuples(index=False):
    path = Path(row.filepath)
    try:
        with Image.open(path) as im:
            im.verify()
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        valid_hashes.append(digest)
    except (UnidentifiedImageError, OSError, ValueError) as exc:
        bad_files.append({"filepath": str(path), "split": row.split, "class_name": row.class_name, "error": str(exc)})
        valid_hashes.append(None)

all_labeled["sha256"] = valid_hashes
bad_df = pd.DataFrame(bad_files)
if not bad_df.empty:
    print("File rusak/tidak terbaca:", len(bad_df))
    display(bad_df.head(20))
else:
    print("Audit integritas: tidak ditemukan file rusak.")

valid_df = all_labeled[all_labeled["sha256"].notna()].copy()
train_df = valid_df[valid_df["split"] == "train"].copy()
test_df = valid_df[valid_df["split"] == "test"].copy()

# Deteksi hash yang muncul di train dan test.
train_hashes = set(train_df["sha256"])
test_hashes = set(test_df["sha256"])
cross_split_hashes = train_hashes.intersection(test_hashes)
print("Hash identik lintas train-test:", len(cross_split_hashes))

if cross_split_hashes:
    before = len(train_df)
    train_df = train_df[~train_df["sha256"].isin(cross_split_hashes)].copy()
    print(f"Mengeluarkan {before - len(train_df)} salinan dari sisi training untuk mencegah exact-duplicate leakage.")

# Duplikasi dalam masing-masing split hanya dilaporkan, tidak dihapus otomatis.
train_dup_count = int(train_df.duplicated("sha256").sum())
test_dup_count = int(test_df.duplicated("sha256").sum())
print("Duplikasi tambahan dalam train:", train_dup_count)
print("Duplikasi tambahan dalam test :", test_dup_count)

if len(train_df) == 0 or len(test_df) == 0:
    raise RuntimeError("Manifest train/test kosong setelah audit. Periksa struktur dataset.")

print("\nJumlah valid train:", len(train_df))
print("Jumlah valid test :", len(test_df))
print("Jumlah valid total:", len(train_df) + len(test_df))


In [ ]:
# Distribusi kelas aktual sesudah audit
train_counts = train_df["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0)
test_counts = test_df["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0)
class_summary = pd.DataFrame({"train_original": train_counts, "test": test_counts})
class_summary["total_labeled"] = class_summary.sum(axis=1)
display(class_summary)

ax = class_summary[["train_original", "test"]].plot(kind="bar", figsize=(10, 5))
ax.set_title("Distribusi kelas dataset setelah audit")
ax.set_xlabel("Kelas")
ax.set_ylabel("Jumlah gambar")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

print("Rasio kelas train terbesar / terkecil:", round(train_counts.max() / train_counts[train_counts > 0].min(), 3))


## 5. Split stratified dan pipeline `tf.data`

Data `seg_train` dibagi menjadi 80% training dan 20% validation. Test set asli tidak disentuh pada proses split. Gambar didekode sebagai RGB dan diubah ukurannya menjadi 150×150. Normalisasi spesifik model dilakukan di dalam arsitektur agar proses inference aplikasi menggunakan preprocessing yang sama.

In [ ]:
SEED = 42
IMAGE_SIZE = (150, 150)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Mengunci seed Python, NumPy, dan TensorFlow.
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

train_part_df, val_df = train_test_split(
    train_df,
    test_size=0.20,
    random_state=SEED,
    stratify=train_df["label"],
)
train_part_df = train_part_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Training   : {len(train_part_df):,}")
print(f"Validation : {len(val_df):,}")
print(f"Testing    : {len(test_df):,}")
print("\nDistribusi training/validation/test:")
split_counts = pd.DataFrame({
    "train": train_part_df["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0),
    "validation": val_df["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0),
    "test": test_df["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0),
})
display(split_counts)


def decode_resize(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE, antialias=True)
    image = tf.cast(image, tf.float32)  # rentang tetap sekitar 0-255
    return image, tf.cast(label, tf.int32)


def make_dataset(frame: pd.DataFrame, training: bool = False):
    paths = frame["filepath"].astype(str).to_numpy()
    labels = frame["label"].astype("int32").to_numpy()
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_resize, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_part_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)

print("Pipeline tf.data siap.")


In [ ]:
# Visualisasi contoh gambar hasil decode dan resize
images, labels = next(iter(train_ds))
plt.figure(figsize=(12, 7))
for i in range(min(12, len(images))):
    ax = plt.subplot(3, 4, i + 1)
    plt.imshow(tf.cast(tf.clip_by_value(images[i], 0, 255), tf.uint8).numpy())
    plt.title(CLASS_NAMES[int(labels[i].numpy())])
    plt.axis("off")
plt.suptitle("Contoh gambar setelah preprocessing dasar")
plt.tight_layout()
plt.show()


## 6. Menyiapkan penyimpanan hasil ke Google Drive

Model dan hasil eksperimen disimpan ke Google Drive agar tidak hilang saat sesi Colab terputus. Folder output ini yang nantinya menjadi sumber file model untuk aplikasi deployment.

In [ ]:
drive.mount("/content/drive")

OUTPUT_DIR = Path("/content/drive/MyDrive/landscape_dl_project")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
MODEL_DIR = OUTPUT_DIR / "models"
RESULT_DIR = OUTPUT_DIR / "results"

for folder in [OUTPUT_DIR, CHECKPOINT_DIR, MODEL_DIR, RESULT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

with open(RESULT_DIR / "class_names.json", "w", encoding="utf-8") as f:
    json.dump(CLASS_NAMES, f, ensure_ascii=False, indent=2)

# Simpan manifest audit (tanpa menyalin gambar dataset).
train_part_df.to_csv(RESULT_DIR / "train_manifest.csv", index=False)
val_df.to_csv(RESULT_DIR / "validation_manifest.csv", index=False)
test_df.to_csv(RESULT_DIR / "test_manifest.csv", index=False)
class_summary.to_csv(RESULT_DIR / "dataset_class_summary.csv")
print("Output directory:", OUTPUT_DIR)


## 7. Definisi augmentasi dan Custom CNN

**Baseline Custom CNN:** tiga blok convolution dengan 32, 64, dan 128 filter. Setiap blok memiliki dua Conv2D 3×3 + Batch Normalization + ReLU, diikuti MaxPooling. Global Average Pooling mengurangi jumlah parameter dibanding Flatten besar; Dense 128 dan Dropout 0,4 menjadi classifier.

- Adam `learning_rate=0.001` sebagai baseline optimizer yang umum dan stabil.
- Sparse categorical cross-entropy karena label disimpan sebagai integer 0–5.
- Softmax enam output untuk probabilitas kelas.
- E1 tanpa augmentasi; E2 dengan augmentasi ringan.


In [ ]:
def make_augmentation(name="data_augmentation"):
    return keras.Sequential(
        [
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.05),
            layers.RandomZoom(0.10),
            layers.RandomContrast(0.10),
        ],
        name=name,
    )


def build_custom_cnn(use_augmentation: bool):
    inputs = keras.Input(shape=(*IMAGE_SIZE, 3), name="image_0_255")
    x = make_augmentation("custom_augmentation")(inputs) if use_augmentation else inputs
    x = layers.Rescaling(1.0 / 255.0, name="custom_rescaling_0_1")(x)

    for block_index, filters in enumerate([32, 64, 128], start=1):
        # Dua convolution per blok mempelajari pola visual bertingkat.
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False,
                          kernel_initializer="he_normal", name=f"block{block_index}_conv1")(x)
        x = layers.BatchNormalization(name=f"block{block_index}_bn1")(x)
        x = layers.Activation("relu", name=f"block{block_index}_relu1")(x)

        x = layers.Conv2D(filters, 3, padding="same", use_bias=False,
                          kernel_initializer="he_normal", name=f"block{block_index}_conv2")(x)
        x = layers.BatchNormalization(name=f"block{block_index}_bn2")(x)
        x = layers.Activation("relu", name=f"block{block_index}_relu2")(x)

        x = layers.MaxPooling2D(pool_size=2, name=f"block{block_index}_pool")(x)
        x = layers.Dropout(0.10 + 0.05 * (block_index - 1), name=f"block{block_index}_dropout")(x)

    x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    x = layers.Dense(128, activation="relu", name="classifier_dense")(x)
    x = layers.Dropout(0.40, name="classifier_dropout")(x)
    outputs = layers.Dense(len(CLASS_NAMES), activation="softmax", name="class_probabilities")(x)

    return keras.Model(inputs, outputs, name=f"custom_cnn_aug_{str(use_augmentation).lower()}")


def compile_model(model, learning_rate):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    return model


model_check = compile_model(build_custom_cnn(use_augmentation=True), 1e-3)
model_check.summary()
del model_check
gc.collect()


## 8. Definisi MobileNetV2 untuk transfer learning

MobileNetV2 menggunakan bobot ImageNet dengan `include_top=False`. Input 150×150×3 valid karena ukuran minimum MobileNetV2 lebih kecil dari 150×150. Pixel dinormalisasi ke −1…1 di dalam model (setara dengan `mobilenet_v2.preprocess_input` untuk input 0…255).

- E3: backbone dibekukan, classifier baru dilatih (feature extraction).
- E4: mulai dari checkpoint E3; 30 layer terakhir backbone dibuka, tetapi Batch Normalization tetap dibekukan. Learning rate diturunkan menjadi `1e-5` untuk menjaga bobot pretrained.

In [ ]:
def build_mobilenetv2(use_augmentation=True):
    inputs = keras.Input(shape=(*IMAGE_SIZE, 3), name="image_0_255")
    x = make_augmentation("mobilenet_augmentation")(inputs) if use_augmentation else inputs
    # MobileNetV2 mengharapkan input dalam rentang -1 sampai 1.
    x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name="mobilenet_preprocessing_minus1_1")(x)

    backbone = keras.applications.MobileNetV2(
        input_shape=(*IMAGE_SIZE, 3),
        alpha=1.0,
        include_top=False,
        weights="imagenet",
    )
    backbone.trainable = False
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D(name="mobilenet_global_average_pooling")(x)
    x = layers.Dropout(0.30, name="mobilenet_dropout")(x)
    x = layers.Dense(128, activation="relu", name="mobilenet_classifier_dense")(x)
    outputs = layers.Dense(len(CLASS_NAMES), activation="softmax", name="class_probabilities")(x)
    return keras.Model(inputs, outputs, name="mobilenetv2_transfer_learning")

# Inisialisasi dan pemeriksaan bentuk model; mengunduh bobot ImageNet jika belum tersedia.
mobile_check = compile_model(build_mobilenetv2(use_augmentation=True), 1e-3)
mobile_check.summary()
print("Jumlah parameter trainable:", sum(np.prod(v.shape) for v in mobile_check.trainable_variables))
del mobile_check
gc.collect()


## 9. Callback dan fungsi training

Checkpoint menyimpan model dengan validation loss terbaik. Early stopping menghentikan training jika validation loss tidak membaik, sedangkan ReduceLROnPlateau menurunkan learning rate ketika proses mulai stagnan.

In [ ]:
def make_callbacks(experiment_name: str, patience: int = 5):
    checkpoint_path = CHECKPOINT_DIR / f"{experiment_name}.keras"
    csv_path = RESULT_DIR / f"{experiment_name}_history.csv"
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(checkpoint_path), monitor="val_loss",
            save_best_only=True, mode="min", verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience,
            restore_best_weights=True, verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.2, patience=2,
            min_lr=1e-6, verbose=1,
        ),
        keras.callbacks.CSVLogger(str(csv_path)),
    ]
    return callbacks, checkpoint_path, csv_path


def fit_experiment(experiment_name, model, epochs, learning_rate, patience=5):
    print(f"\n===== {experiment_name} =====")
    model = compile_model(model, learning_rate)
    callbacks, checkpoint_path, csv_path = make_callbacks(experiment_name, patience)
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1,
    )
    history_df = pd.DataFrame(history.history)
    history_df.to_csv(csv_path, index=False)
    print("Checkpoint terbaik:", checkpoint_path)
    return history_df, checkpoint_path


## 10. Eksperimen E1 — Custom CNN tanpa augmentasi (baseline)

Maksimal 30 epoch, Adam `1e-3`. Ini mengukur kemampuan CNN kustom tanpa augmentasi.

In [ ]:
history_e1, path_e1 = fit_experiment(
    experiment_name="E1_custom_cnn_no_augmentation",
    model=build_custom_cnn(use_augmentation=False),
    epochs=30,
    learning_rate=1e-3,
    patience=5,
)


## 11. Eksperimen E2 — Custom CNN dengan augmentasi

Arsitektur, split, optimizer, dan learning rate sama dengan E1. Perbedaan utama hanya augmentasi pada training sehingga pengaruh augmentasi dapat diamati.

In [ ]:
history_e2, path_e2 = fit_experiment(
    experiment_name="E2_custom_cnn_with_augmentation",
    model=build_custom_cnn(use_augmentation=True),
    epochs=30,
    learning_rate=1e-3,
    patience=5,
)


## 12. Eksperimen E3 — MobileNetV2 feature extraction

Backbone ImageNet dibekukan; hanya classifier baru yang dilatih. Augmentasi diaktifkan. Maksimal 15 epoch, Adam `1e-3`.

In [ ]:
history_e3, path_e3 = fit_experiment(
    experiment_name="E3_mobilenetv2_feature_extraction",
    model=build_mobilenetv2(use_augmentation=True),
    epochs=15,
    learning_rate=1e-3,
    patience=4,
)


## 13. Eksperimen E4 — MobileNetV2 fine-tuning

E4 dilanjutkan dari checkpoint terbaik E3. Tiga puluh layer terakhir backbone dibuka, sedangkan layer Batch Normalization tetap dibekukan. Model **wajib di-compile ulang** setelah pengaturan `trainable` diubah.

In [ ]:
# Muat checkpoint E3 terbaik sebagai titik awal fine-tuning.
finetune_model = keras.models.load_model(path_e3)
# Temukan nested MobileNetV2 model berdasarkan nama layer bawaan Keras.
backbone = next(
    layer for layer in finetune_model.layers
    if isinstance(layer, keras.Model) and layer.name.lower().startswith("mobilenetv2")
)
print("Backbone:", backbone.name)
backbone.trainable = True

# Bekukan semua layer terlebih dahulu, kemudian buka 30 layer terakhir.
for layer in backbone.layers:
    layer.trainable = False
for layer in backbone.layers[-30:]:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True

trainable_backbone = sum(int(layer.trainable) for layer in backbone.layers)
print("Layer backbone yang trainable:", trainable_backbone, "dari", len(backbone.layers))

# Compile ulang dengan learning rate kecil agar bobot pretrained tidak berubah drastis.
history_e4, path_e4 = fit_experiment(
    experiment_name="E4_mobilenetv2_finetuning",
    model=finetune_model,
    epochs=10,
    learning_rate=1e-5,
    patience=4,
)
del finetune_model, backbone
gc.collect()


## 14. Plot training dan validation loss/accuracy

Grafik membantu mengamati overfitting, underfitting, dan kestabilan training. CSV history setiap eksperimen disimpan ke folder `results`.

In [ ]:
EXPERIMENT_FILES = {
    "E1 Custom CNN tanpa augmentasi": RESULT_DIR / "E1_custom_cnn_no_augmentation_history.csv",
    "E2 Custom CNN dengan augmentasi": RESULT_DIR / "E2_custom_cnn_with_augmentation_history.csv",
    "E3 MobileNetV2 feature extraction": RESULT_DIR / "E3_mobilenetv2_feature_extraction_history.csv",
    "E4 MobileNetV2 fine-tuning": RESULT_DIR / "E4_mobilenetv2_finetuning_history.csv",
}

fig, axes = plt.subplots(2, 1, figsize=(12, 10))
for label, csv_path in EXPERIMENT_FILES.items():
    hist = pd.read_csv(csv_path)
    if "accuracy" in hist.columns and "val_accuracy" in hist.columns:
        axes[0].plot(hist["accuracy"], label=f"{label} — train")
        axes[0].plot(hist["val_accuracy"], linestyle="--", label=f"{label} — val")
    if "loss" in hist.columns and "val_loss" in hist.columns:
        axes[1].plot(hist["loss"], label=f"{label} — train")
        axes[1].plot(hist["val_loss"], linestyle="--", label=f"{label} — val")

axes[0].set_title("Accuracy per epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend(fontsize=8, ncol=2)
axes[1].set_title("Loss per epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(RESULT_DIR / "training_curves_all_experiments.png", dpi=160, bbox_inches="tight")
plt.show()


## 15. Model selection berdasarkan macro F1 validation

Setiap checkpoint terbaik (berdasarkan `val_loss`) dievaluasi kembali pada validation split. Pemenang tiap keluarga arsitektur dipilih berdasarkan macro F1 validation. Model terbaik keseluruhan juga ditentukan dari validation, bukan dari test.

In [ ]:
EXPERIMENT_PATHS = {
    "E1_custom_cnn_no_augmentation": path_e1,
    "E2_custom_cnn_with_augmentation": path_e2,
    "E3_mobilenetv2_feature_extraction": path_e3,
    "E4_mobilenetv2_finetuning": path_e4,
}


def get_true_labels(dataset):
    return np.concatenate([labels.numpy() for _, labels in dataset], axis=0)


def metric_dict(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
    }


y_val = get_true_labels(val_ds)
val_rows = []
for experiment_name, model_path in EXPERIMENT_PATHS.items():
    print("Evaluasi validation:", experiment_name)
    candidate = keras.models.load_model(model_path)
    val_prob = candidate.predict(val_ds, verbose=0)
    val_pred = np.argmax(val_prob, axis=1)
    row = {"experiment": experiment_name, **metric_dict(y_val, val_pred)}
    val_rows.append(row)
    del candidate
    gc.collect()
    tf.keras.backend.clear_session()

val_metrics_df = pd.DataFrame(val_rows).sort_values("macro_f1", ascending=False)
display(val_metrics_df)
val_metrics_df.to_csv(RESULT_DIR / "validation_metrics.csv", index=False)

custom_rows = val_metrics_df[val_metrics_df["experiment"].str.startswith(("E1_", "E2_"))]
mobile_rows = val_metrics_df[val_metrics_df["experiment"].str.startswith(("E3_", "E4_"))]
best_custom_name = custom_rows.iloc[0]["experiment"]
best_mobile_name = mobile_rows.iloc[0]["experiment"]
best_overall_name = val_metrics_df.iloc[0]["experiment"]

best_custom_path = MODEL_DIR / "custom_cnn_best.keras"
best_mobile_path = MODEL_DIR / "mobilenetv2_best.keras"
best_overall_path = MODEL_DIR / "best_overall_model.keras"
shutil.copy2(EXPERIMENT_PATHS[best_custom_name], best_custom_path)
shutil.copy2(EXPERIMENT_PATHS[best_mobile_name], best_mobile_path)
shutil.copy2(EXPERIMENT_PATHS[best_overall_name], best_overall_path)

selection = {
    "best_custom_cnn_experiment": best_custom_name,
    "best_mobilenetv2_experiment": best_mobile_name,
    "best_overall_experiment": best_overall_name,
    "selection_metric": "validation_macro_f1",
}
with open(RESULT_DIR / "model_selection.json", "w", encoding="utf-8") as f:
    json.dump(selection, f, ensure_ascii=False, indent=2)
print("Model terpilih:", selection)


## 16. Evaluasi final pada test set

Setelah semua eksperimen dan pemilihan model dikunci berdasarkan validation, evaluasi test dilakukan satu kali untuk pelaporan. Keempat checkpoint dapat dibandingkan secara transparan; jangan melakukan tuning ulang berdasarkan hasil test.

In [ ]:
y_test = get_true_labels(test_ds)
test_rows = []
test_predictions = {}

for experiment_name, model_path in EXPERIMENT_PATHS.items():
    print("Evaluasi test:", experiment_name)
    candidate = keras.models.load_model(model_path)
    test_prob = candidate.predict(test_ds, verbose=0)
    test_pred = np.argmax(test_prob, axis=1)
    test_predictions[experiment_name] = test_pred
    test_rows.append({"experiment": experiment_name, **metric_dict(y_test, test_pred)})
    del candidate
    gc.collect()
    tf.keras.backend.clear_session()

test_metrics_df = pd.DataFrame(test_rows).sort_values("macro_f1", ascending=False)
display(test_metrics_df)
test_metrics_df.to_csv(RESULT_DIR / "test_metrics_all_experiments.csv", index=False)

# Pilih prediksi final berdasarkan model yang sebelumnya menang di validation.
best_test_pred = test_predictions[best_overall_name]
print("\nModel final (dipilih via validation):", best_overall_name)
print(classification_report(y_test, best_test_pred, target_names=CLASS_NAMES, digits=4, zero_division=0))


In [ ]:
# Confusion matrix model terbaik yang dipilih berdasarkan validation macro-F1
cm = confusion_matrix(y_test, best_test_pred, labels=list(range(len(CLASS_NAMES))))
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, interpolation="nearest")
ax.figure.colorbar(im, ax=ax)
ax.set(
    xticks=np.arange(len(CLASS_NAMES)),
    yticks=np.arange(len(CLASS_NAMES)),
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    ylabel="Label sebenarnya",
    xlabel="Prediksi model",
    title=f"Confusion Matrix — {best_overall_name}",
)
plt.setp(ax.get_xticklabels(), rotation=35, ha="right", rotation_mode="anchor")
threshold = cm.max() / 2.0 if cm.size else 0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], "d"), ha="center", va="center",
                color="white" if cm[i, j] > threshold else "black")
fig.tight_layout()
plt.savefig(RESULT_DIR / "confusion_matrix_best_model.png", dpi=160, bbox_inches="tight")
plt.show()


## 17. Analisis kesalahan prediksi

Bagian ini menampilkan contoh gambar test yang salah diprediksi oleh model terbaik. Contoh ini bisa digunakan untuk membahas kemiripan karakteristik visual, misalnya glacier vs mountain atau buildings vs street. Jangan menyimpulkan penyebab hanya dari gambar; hubungkan pengamatan visual dengan confusion matrix dan konteks dataset.

In [ ]:
wrong_idx = np.where(best_test_pred != y_test)[0]
print("Jumlah salah prediksi:", len(wrong_idx), "dari", len(y_test))

if len(wrong_idx) > 0:
    rng = np.random.default_rng(SEED)
    chosen = rng.choice(wrong_idx, size=min(12, len(wrong_idx)), replace=False)
    fig = plt.figure(figsize=(12, 8))
    for plot_i, sample_i in enumerate(chosen, start=1):
        path = test_df.iloc[int(sample_i)]["filepath"]
        img = Image.open(path).convert("RGB").resize(IMAGE_SIZE)
        ax = plt.subplot(3, 4, plot_i)
        ax.imshow(img)
        ax.set_title(f"True: {CLASS_NAMES[y_test[sample_i]]}\nPred: {CLASS_NAMES[best_test_pred[sample_i]]}", fontsize=9)
        ax.axis("off")
    plt.suptitle("Contoh kesalahan prediksi pada test set")
    plt.tight_layout()
    plt.savefig(RESULT_DIR / "misclassified_examples.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("Tidak ada kesalahan prediksi pada test set.")


## 18. Simpan ringkasan eksperimen dan artefak final

File utama yang disimpan:

- `models/custom_cnn_best.keras`
- `models/mobilenetv2_best.keras`
- `models/best_overall_model.keras`
- `results/class_names.json`
- `results/validation_metrics.csv`
- `results/test_metrics_all_experiments.csv`
- grafik training, confusion matrix, dan contoh salah prediksi.

Model `.keras` sudah memuat preprocessing dan augmentasi yang menjadi bagian arsitektur, sehingga aplikasi nantinya harus mengirim gambar RGB 150×150 dengan nilai pixel 0–255 (tanpa normalisasi eksternal).

In [ ]:
# Simpan metadata eksperimen untuk keperluan laporan dan deployment.
summary = {
    "dataset": "Intel Image Classification",
    "source": "https://www.kaggle.com/datasets/puneet6060/intel-image-classification",
    "automatic_download_mirror": "https://huggingface.co/datasets/sfarrukhm/intel-image-classification",
    "download_method": "huggingface_hub.snapshot_download (public mirror; no kaggle.json upload)",
    "classes": CLASS_NAMES,
    "image_size": list(IMAGE_SIZE),
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "train_count": int(len(train_part_df)),
    "validation_count": int(len(val_df)),
    "test_count": int(len(test_df)),
    "experiments": {
        "E1": {"model": "Custom CNN", "augmentation": False, "max_epochs": 30, "learning_rate": 1e-3},
        "E2": {"model": "Custom CNN", "augmentation": True, "max_epochs": 30, "learning_rate": 1e-3},
        "E3": {"model": "MobileNetV2 feature extraction", "augmentation": True, "max_epochs": 15, "learning_rate": 1e-3},
        "E4": {"model": "MobileNetV2 fine-tuning", "augmentation": True, "max_epochs": 10, "learning_rate": 1e-5, "unfrozen_backbone_layers": 30},
    },
    "model_selection": selection,
    "test_metrics": test_rows,
}
with open(RESULT_DIR / "experiment_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\nSelesai. Artefak tersimpan di:", OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(OUTPUT_DIR)} — {p.stat().st_size / (1024*1024):.2f} MB")


In [ ]:
# 19. Buat satu ZIP artefak untuk menghubungkan hasil Colab ke aplikasi Streamlit
# ZIP ini hanya berisi dua model terbaik dan artefak evaluasi yang dibaca aplikasi.
APP_BUNDLE = OUTPUT_DIR / "landscape_ai_classifier_artifacts.zip"
BUNDLE_TEMP = OUTPUT_DIR / "_app_bundle_temp"
if BUNDLE_TEMP.exists():
    shutil.rmtree(BUNDLE_TEMP)
(BUNDLE_TEMP / "models").mkdir(parents=True, exist_ok=True)
(BUNDLE_TEMP / "results").mkdir(parents=True, exist_ok=True)

required_models = [
    MODEL_DIR / "custom_cnn_best.keras",
    MODEL_DIR / "mobilenetv2_best.keras",
]
missing_models = [str(p) for p in required_models if not p.exists()]
if missing_models:
    raise FileNotFoundError("Model deployment belum lengkap: " + ", ".join(missing_models))

for model_path in required_models:
    shutil.copy2(model_path, BUNDLE_TEMP / "models" / model_path.name)
overall_model = MODEL_DIR / "best_overall_model.keras"
if overall_model.exists():
    shutil.copy2(overall_model, BUNDLE_TEMP / "models" / overall_model.name)

# Hanya file yang dibutuhkan oleh aplikasi, bukan manifest path seluruh dataset.
APP_RESULT_FILES = [
    "class_names.json",
    "dataset_class_summary.csv",
    "validation_metrics.csv",
    "test_metrics_all_experiments.csv",
    "training_curves_all_experiments.png",
    "confusion_matrix_best_model.png",
    "model_selection.json",
    "experiment_summary.json",
    "E1_custom_cnn_no_augmentation_history.csv",
    "E2_custom_cnn_with_augmentation_history.csv",
    "E3_mobilenetv2_feature_extraction_history.csv",
    "E4_mobilenetv2_finetuning_history.csv",
]
missing_results = [name for name in APP_RESULT_FILES if not (RESULT_DIR / name).exists()]
if missing_results:
    raise FileNotFoundError("Artefak hasil belum lengkap: " + ", ".join(missing_results))
for name in APP_RESULT_FILES:
    shutil.copy2(RESULT_DIR / name, BUNDLE_TEMP / "results" / name)
# Contoh gambar salah prediksi bersifat opsional jika model tidak memiliki error pada test.
optional_error_plot = RESULT_DIR / "misclassified_examples.png"
if optional_error_plot.exists():
    shutil.copy2(optional_error_plot, BUNDLE_TEMP / "results" / optional_error_plot.name)

with zipfile.ZipFile(APP_BUNDLE, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for file_path in sorted(BUNDLE_TEMP.rglob("*")):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(BUNDLE_TEMP))

print("ZIP artefak aplikasi dibuat:", APP_BUNDLE)
print("Ukuran (MB):", round(APP_BUNDLE.stat().st_size / (1024 * 1024), 2))
print("Isi ZIP:")
with zipfile.ZipFile(APP_BUNDLE, "r") as zf:
    for name in zf.namelist():
        print("-", name)

# Colab mengunduh satu ZIP hasil secara otomatis (bukan upload / Choose Files).
from google.colab import files
files.download(str(APP_BUNDLE))


## 20. Pemeriksaan akhir sebelum Tahap 3

Sebelum menghubungkan model ke Streamlit, pastikan:

1. Audit dataset selesai dan jumlah file aktual tercatat.
2. Empat eksperimen E1–E4 selesai atau berhenti dengan EarlyStopping.
3. `custom_cnn_best.keras` dan `mobilenetv2_best.keras` tersedia.
4. `test_metrics_all_experiments.csv`, `validation_metrics.csv`, `training_curves_all_experiments.png`, `confusion_matrix_best_model.png`, `class_names.json`, `model_selection.json`, dan `experiment_summary.json` tersedia.
5. ZIP `landscape_ai_classifier_artifacts.zip` selesai diunduh dari output Colab.

### Cara memasang artefak ke aplikasi
Ekstrak ZIP artefak dari Colab ke folder utama `landscape_ai_classifier_ui/` (folder `models/` dan `results/` akan digabungkan). Jangan menaruh file model ke GitHub jika ukuran file terlalu besar atau aturan repository tidak mengizinkan; gunakan Git LFS atau penyimpanan artefak.

**Penting:** angka evaluasi harus berasal dari run ini. Jangan mengarang akurasi. Dataset mentah tidak masuk ZIP artefak aplikasi dan tidak perlu diunggah ke GitHub.

### Sumber
- Intel Image Classification — Kaggle: https://www.kaggle.com/datasets/puneet6060/intel-image-classification
- Mirror publik untuk download otomatis: https://huggingface.co/datasets/sfarrukhm/intel-image-classification
- Hugging Face Hub `snapshot_download`: https://huggingface.co/docs/huggingface_hub/guides/download
- TensorFlow MobileNetV2: https://www.tensorflow.org/api_docs/python/tf/keras/applications/MobileNetV2
- Transfer learning dan fine-tuning: https://www.tensorflow.org/guide/keras/transfer_learning
